# Aegis Command - SOTA Vision Model Fine-Tuning (Colab GPU Pipeline)

This notebook fine-tunes a State-Of-The-Art YOLOv8 / YOLOv11 model on the Aegis Command aerial drone dataset using a free Google Colab GPU (T4/V100/A100).

### Dataset Overview:
- 155 high-resolution (1920x1080) drone frames
- 1,948 human-verified ground-truth bounding boxes (`person`, `car`, `truck`, `bus`)
- Stratified 80/20 train/validation split (124 train / 31 holdout val)

In [ ]:
# 1. Check GPU availability
!nvidia-smi

In [ ]:
# 2. Install Ultralytics and dependencies
%pip install -q ultralytics albumentations opencv-python matplotlib

In [ ]:
# 3. Clone Repository or Upload Dataset
!git clone https://github.com/TanmayJain-dev/Aegis-Command.git
%cd Aegis-Command
!git checkout feat/accuracy-v2

In [ ]:
# 4. Prepare Dataset Splits
!python vision/eval/split_dataset.py

In [ ]:
# 5. Run GPU Fine-Tuning (50 Epochs, Multi-scale 1280px, Mosaic Data Augmentation)
from ultralytics import YOLO

# Choose model scale: yolov8s.pt, yolov8m.pt, or yolo11s.pt
model = YOLO('yolov8s.pt')

results = model.train(
    data='vision/data/aegis_data.yaml',
    epochs=50,
    imgsz=1280,
    batch=16,
    device=0,
    workers=4,
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    box=7.5,
    cls=0.5,
    project='runs/train',
    name='aegis_sota_gpu',
    save=True,
    plots=True
)

In [ ]:
# 6. Evaluate On Held-Out Validation Split
!python vision/eval/eval_detection.py \
    --model runs/train/aegis_sota_gpu/weights/best.pt \
    --split val \
    --imgsz 1280 \
    --out runs/eval/gpu_eval_results.json

In [ ]:
# 7. Download Trained Weights
from google.colab import files
files.download('runs/train/aegis_sota_gpu/weights/best.pt')